# Attention is all You Need

- This is the name of the paper that introduced the transformer. Linked [here](https://arxiv.org/abs/1706.03762)
- We will be training the transformer model on the [Tiny Shakespeare dataset](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt). This is a concatenation of all of shakesepeare's works. The idea is that the transformer model will look at prior characters and will predict the next character in the sequence. The transformer will just try to produce the correct sequences
- Characters are being released on a token-by-token level (tokens are subword pieces)
-nanoGPT is used to train the transformers on the given text. This is composed of two files where one file defines the GPT model and the other trains the model on some given text dataset

In [1]:
from structures.config import get_params
import requests

## Reading and Exploring the Data

In [2]:
# Define a function to get text data from a url
def get_text(url: str) -> str:
    response = requests.get(url)
    if response.status_code == 200:
        byte_data = response.content
        text = byte_data.decode('utf-8')
        return text

In [3]:
params = get_params()
text_data = get_text(params.tiny_shakespeare)

In [4]:
len(text_data) # Number of characters in the text

1115394

In [5]:
# unique characters in the text, sorted
characters = sorted(list(set(text_data))) # Convert to set so that we only have a list of unique characters, then convert to list and sort for ordering
vocab_size = len(characters) # Gives the number of unique characters we have
print(vocab_size) # Prints out the number of characters in our text data
print(''.join(characters)) # Prints out each character that appearas in our text data

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


## Tokenization, Train/Val Split

In [6]:
# Creating an encoder and decoder - this translates characters to integers and vice versa

# The below code depicts a simple example of an encoder and decoder. There are many others. Our encoding/decoding works on the character level
## Other encoders and decoders, such as Google's SentencePiece will not use entire words nor individual characters, rather sub-words. This is the standard practice
## OpenAI uses Byte-pair encoding tokenizer called 'tiktoken' 
char_to_int = {character:idx for idx, character in enumerate(characters)}
int_to_char = {idx:character for idx, character in enumerate(characters)}

encoder = lambda s: [char_to_int[character] for character in s] # Encodes the character into a value
decoder = lambda l: ''.join([int_to_char[integer] for integer in l]) # Decodes the values to get the characters

print(encoder('Hello, World!')) # Gives back some list of numbers
print(decoder(encoder('Hello, World!'))) # Should give back 'Hello, World!'

[20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Hello, World!


In [7]:
# Example using tiktoken
import tiktoken

enc = tiktoken.get_encoding('gpt2') # Getting the GPT-2 encoder
print(enc.n_vocab) # Gives the number of possible tokens in gpt-2's encoder. GPT-2 has tokens with values from 0 to 50256!

# You can trade off between code book size and sequence lengths! 
## This means you can have very long sequences of integers with very small vocabularies or very large sequences of integers with very small vocabularies
## The encoder/decoder we made earlier is very simple. We have very simple encode and decode functions but very large sequences of integers as a result

encoded_tiktoken = enc.encode('Hello, World!')
print(encoded_tiktoken)
decoded_tiktoken = enc.decode(encoded_tiktoken)
print(decoded_tiktoken)

50257
[15496, 11, 2159, 0]
Hello, World!


In [8]:
# We can now tokenize the entire training set of shakespeare
import torch
data = torch.tensor(encoder(text_data), dtype=torch.long) #wrapping our values in a pytorch tensor
print(data.shape, data.dtype)
print(data[:1000]) #This is the first 1000 characters being fed into GPT

c:\Users\omara\Desktop\workspace\gpt_from_scratch\venvironment\Lib\site-packages\torch\nn\modules\transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at ..\torch\csrc\utils\tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
# Separate our data into train and validation (90-10 split)
n = int(.9 * len(data))
train_data = data[:n]
val_data = data[n:] # We withhold some amount of data to keep the neural net from memorizing the data. That's because we effectively want to be able to generate shakespeareian text!

## Data Loader: Batches of chunks of data

We don't want to feed the data in all at once to the transformer, as it's very computationally expensive. Instead we feed it training data in random chunks.  
There's usually a maximum length to the chunk size of the data! This is usually denoted as "block size"

In [11]:
block_size = 8
train_data[:block_size+1] # Gives the first 9 characters of the training set

## IMPORTANT NOTE: It is important to understand the following - We are predicting sequences of characters. 
### Therefore, whatever chunk size of training data we have, the corresponding predicted value is character that follows that chunk. 
### For example: if we have the following chunk of characters [18, 47, 56, 57], then the value we are predicting from the first 3 characters is the 4th character! 
### So for a chunk of 9 characters we have 8 predicters!

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
x = train_data[:block_size]
y = train_data[1:block_size+1] # Offsetting everything by 1
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

# The below output is the chunk of 8 examples hidden in the 9 characters!
## This is done to make the transformer accustomed to seeing context from as little as 9 character all the way up to the block size. We want the transformer to be used to seeing everything in between
## This is going to be useful later during inference because while we sample we can start the sampling duration with as little as one character of context and then it can start predicting from one character
## all the way up to the block size.
## After the block size we start truncating because the transformer will never receive more than the block size in inputs when it's predicting its next character.
## This all encompasses the time dimension of a transformer model

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
# The batch dimension
## As we're sampling these chunks of text, every time we're going to be feeding them into a transformer we're going to have many batches of multiple chunks of text that are all stacked in a single tensor
## This is done for efficiency because of their parallel processing capabilities
## We process multiple chunks at the same time. They're processed independently


torch.manual_seed(1337) # Sets the seed for generating random numbers
batch_size = 4 # How many independent sequences we'll run in parallel
block_size = 8 # What is the maximum context length for predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data 
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('outputs:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time (sequence) dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")